## Import related functions

In [1]:
import cobra
import re 
#from script.ECMpy_function import *
import sys
sys.path.append(r'./script/')
from ECMpy_function import *

D:\pycharm\ecCG827\.venv\Lib\site-packages\bioservices\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Input and output files

In [2]:
# reaction kcat_mw
sbml_path = "iCZ870_CGXII.xml"
taxonom_id=1079988
method='DLKcat'#DLKcat
reaction_kcat_MW_file = f'./get_kcat_mw_by_{method}/reaction_kcat_MW.csv'
# #paxdb丰度数据
# gene_abundance_colname='abundance'
# gene_abundance_file='./data/gene_abundance.csv' # downolad from https://pax-db.org/download
#The enzyme mass fraction,such as 0.405
#f=calculate_f_v2(sbml_path, gene_abundance_file,gene_abundance_colname,taxonom_id)
f=0.46
#Initial parameters
ptot = 0.56 # The total protein fraction in cell.
sigma = 0.5 # The approximated saturation of enzyme.e.g.,0.5/1.
lowerbound = 0   # Lowerbound  of enzyme concentration constraint. 
upperbound = round(ptot * f * sigma, 3)#total enzyme
ecModel_output_file=f'./model/ecCZ870_{method}.json'

## Get ecModel and simulation

In [4]:
#Get ecModel
trans_model2enz_json_model_split_isoenzyme(sbml_path, reaction_kcat_MW_file, f, ptot, sigma, lowerbound, upperbound, ecModel_output_file)

#ecModel Simulation
obj='Bio_cgATCC14067_c'#
fluxes_outfile = './get_kcat_mw_by_%s/ECMpy_solution_%s_pfba.csv' % (method, obj)
use_substrate='EX_cpd00027_e'
concentration=5.1
enz_model=get_enzyme_constraint_model(ecModel_output_file)
enz_model.objective=obj

#change original substrate in model
[ori_obj_id,ori_substrate_id_list,ori_sub_concentration,ori_ATPM]=get_model_substrate_obj(enz_model)
for eachsubid in ori_substrate_id_list:
    if re.search('_reverse',eachsubid):
        r_id_new=eachsubid.split('_reverse')[0]
        enz_model.reactions.get_by_id(eachsubid).bounds = (0, 0) 
        enz_model.reactions.get_by_id(r_id_new).bounds = (0, 0)  
    else:
        r_id_new=eachsubid+'_reverse'
        enz_model.reactions.get_by_id(eachsubid).bounds = (0, 0) 
        enz_model.reactions.get_by_id(r_id_new).bounds = (0, 0) 

enz_model.reactions.get_by_id(use_substrate).bounds = (-concentration, 0)
enz_model.reactions.get_by_id(use_substrate+'_reverse').bounds = (0, 0)

enz_model_pfba_solution = cobra.flux_analysis.pfba(enz_model)
enz_model_pfba_solution = get_fluxes_detail_in_model(enz_model,enz_model_pfba_solution,fluxes_outfile,ecModel_output_file)
print(enz_model_pfba_solution.fluxes[obj])

Set parameter Username
Set parameter LicenseID to value 2723056
Academic license - for non-commercial use only - expires 2026-10-16


No objective coefficients in model. Unclear what should be optimized


0.47567297116529556


In [3]:
reaction_kcat_MW_file = f'./model_Calibration/{method}_Calibration/reaction_kcat_MW_modify.csv'
ecModel_output_file=f'./model_Calibration/{method}_Calibration/ecCZ870_{method}_Calibration_modify.json'